# 🧬 Notebook 3B — Large Custom DNA Transformer Across All Tokenizers

## One architecture. Five DNA representations. Multi-GPU training.

Notebook 2 taught us how to **build a Transformer from scratch** and how DNA can be represented in several ways.

Notebook 3B scales that idea up.

We will train a **fresh larger custom Transformer** for each tokenizer:

```text
1. Single nucleotide
2. One-hot
3. Overlapping 6-mer
4. Non-overlapping 6-mer
5. BPE
```

Every run uses the same cleaned dataset, train/validation split, architecture, optimizer settings, epoch count, BF16 precision, and DDP strategy.

The main variable is therefore:

> **How we represent the DNA sequence.**

## Learning objectives

By the end of Notebook 3B, you should be able to:

1. Reuse the five tokenization methods from Notebook 2.
2. Explain why token count changes Transformer compute.
3. Scale a custom Transformer to a larger architecture.
4. Train that architecture with multiple GPUs using DDP.
5. Use BF16 mixed precision on the GPU.
6. Train **one fresh model per tokenizer**.
7. Compare tokenizers using:
   - validation AUROC,
   - AUPRC,
   - training time,
   - examples/second,
   - peak GPU memory,
   - parameter count,
   - mean token count.
8. Decide which tokenizer gives the best balance between biological performance and computational cost.

# 1. Connection to Notebook 2

Notebook 2 used the following representations:

| Tokenizer | Representation | Approximate tokens for 200 bp |
|---|---|---:|
| Single nucleotide | A, C, G, T as IDs | 200 |
| One-hot | one 4D vector per base | 200 |
| Overlapping 6-mer | sliding 6-base words | 195 |
| Non-overlapping 6-mer | chunks of 6 bases | 33 |
| BPE | learned variable-length DNA chunks | variable |

We keep the same important rules:

- PAD ID = `0` for discrete tokenizers.
- Single nucleotide real IDs do not collide with PAD.
- One-hot uses 4 dimensions: A/C/G/T.
- Overlapping 6-mers use all sliding windows.
- Non-overlapping 6-mers discard the final incomplete remainder in this implementation.
- BPE is learned from **training DNA only**.
- BPE uses PAD = 0, UNK = 1, real tokens beginning at 2.
- Dynamic padding pads only to the longest sequence **in the current batch**.

# 2. Why Token Count Matters More for a Larger Transformer

Self-attention compares token positions with other token positions.

A useful conceptual proxy is:

```text
attention work ∝ token_count²
```

For example:

```text
200 tokens → about 40,000 pairwise positions
 33 tokens → about  1,089 pairwise positions
```

This is **not an exact runtime formula**, but it explains why tokenizer choice can dramatically change the compute required by attention.

Notebook 3B lets us measure that consequence on the GPU.

# 3. Setup

In [ ]:
import os
import sys
import json
import time
import shlex
import subprocess
import py_compile

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    os.path.expanduser("~")
)

DATA_DIR = (
    PROJECT_DIR
    / "ctcf_k562_example"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "notebook3b_results"
)

SCRIPTS_DIR = (
    PROJECT_DIR
    / "notebook3b_scripts"
)

SLURM_LOG_DIR = (
    RESULTS_DIR
    / "slurm_logs"
)

for directory in [
    RESULTS_DIR,
    SCRIPTS_DIR,
    SLURM_LOG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

NOTEBOOK_PYTHON = sys.executable

print("Notebook Python:", NOTEBOOK_PYTHON)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

# 4. Large-Model Configuration

Notebook 2 used a deliberately small teaching model.

Notebook 3B uses a substantially larger model while keeping the architecture recognizable.

The default is:

```text
embedding dimension = 256
attention heads     = 8
Transformer layers  = 8
feed-forward width  = 4 × embedding dimension
```

Every parameter in the custom model is trainable because the model starts from scratch.

## Throughput-first batch strategy

Like Notebook 3A, we specify a **local batch per GPU**.

For example:

```text
2 GPUs × 32 examples/GPU = global batch 64
```

This gives each GPU a meaningful amount of work.

In [ ]:
# ✏️ EDIT ME — hardware and architecture

NERSC_ACCOUNT = "m4388"

# Current testing value.
SLURM_QOS = "shared"

GPU_COUNT = 2

# Same local batch for all tokenizers.
LOCAL_BATCH_SIZE = 32

MODEL_CONFIG = {
    "n_embd": 256,
    "n_head": 8,
    "n_layer": 8,
    "dropout": 0.10,

    "epochs": 10,
    "local_batch_size": LOCAL_BATCH_SIZE,

    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,

    "precision": "bf16",
    "bpe_merges": 80,
    "seed": 42,
}

TOKENIZER_ORDER = [
    "single_nucleotide",
    "one_hot",
    "overlap_6mer",
    "nonoverlap_6mer",
    "bpe",
]

pd.Series(MODEL_CONFIG, name="value")

In [ ]:
# 🔒 RUN ONLY — configuration checks

if MODEL_CONFIG["n_embd"] % MODEL_CONFIG["n_head"] != 0:
    raise ValueError(
        "n_embd must be divisible by n_head."
    )

if GPU_COUNT < 1:
    raise ValueError(
        "Request at least one GPU."
    )

if SLURM_QOS == "shared" and GPU_COUNT > 2:
    raise ValueError(
        "Current shared-QOS testing should use 1 or 2 GPUs. "
        "Change the QOS when the bootcamp allocation is available."
    )

print("GPUs:", GPU_COUNT)
print("Local batch/GPU:", LOCAL_BATCH_SIZE)
print("Global batch:", GPU_COUNT * LOCAL_BATCH_SIZE)
print("Tokenizers:", TOKENIZER_ORDER)

# 5. The Larger Custom Transformer

We preserve the architecture students built in Notebook 2:

```text
input representation
↓
token embedding OR one-hot linear projection
↓
learned positional embedding
↓
8 pre-LayerNorm Transformer blocks
↓
final LayerNorm
↓
masked mean pooling
↓
binary classifier
```

Each block still contains:

```text
multi-head self-attention
+
feed-forward network
+
residual connections
+
LayerNorm
```

The model is larger, but the logic is the same.

## Attention scaling remains correct

For each attention head:

```python
scale = head_size ** -0.5
```

We scale by the **head dimension**, not the total embedding dimension.

This is the same correctness rule used in Notebook 2.

# 6. GPU Strategy

Each tokenizer is trained as a **separate fresh model**, but the five models are trained sequentially inside one GPU allocation.

For one tokenizer:

```text
rank 0 / GPU 0 ─┐
                 ├─ DDP gradient all-reduce → one model
rank 1 / GPU 1 ─┘
```

Then that model is saved and released from memory before the next tokenizer begins.

This avoids waiting in the queue five separate times while still giving each tokenizer run the full requested multi-GPU allocation.

## BF16

All five models use BF16 autocast for compatible GPU operations.

The model parameters themselves remain standard PyTorch trainable parameters; autocast chooses lower precision for eligible operations during forward computation.

# 7. Write the Shared DDP Helper

In [ ]:
# 🔒 RUN ONLY — write ddp_common.py

DDP_COMMON_SCRIPT = SCRIPTS_DIR / "ddp_common.py"
DDP_COMMON_SOURCE = '\nimport os\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\n\nfrom sklearn.metrics import (\n    confusion_matrix,\n    precision_score,\n    recall_score,\n    f1_score,\n    roc_auc_score,\n    average_precision_score,\n)\n\n\ndef setup_distributed():\n    rank = int(os.environ.get("RANK", "0"))\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n    world_size = int(os.environ.get("WORLD_SIZE", "1"))\n\n    distributed = world_size > 1\n    visible_gpu_count = torch.cuda.device_count()\n\n    print(\n        f"[DDP setup] RANK={rank} | "\n        f"LOCAL_RANK={local_rank} | "\n        f"WORLD_SIZE={world_size} | "\n        f"visible_GPUs={visible_gpu_count} | "\n        f"CUDA_VISIBLE_DEVICES="\n        f"{os.environ.get(\'CUDA_VISIBLE_DEVICES\', \'<not set>\')}",\n        flush=True,\n    )\n\n    if visible_gpu_count < world_size:\n        raise RuntimeError(\n            f"Rank {rank} sees only {visible_gpu_count} GPU(s), "\n            f"but WORLD_SIZE={world_size}."\n        )\n\n    if local_rank >= visible_gpu_count:\n        raise RuntimeError(\n            f"LOCAL_RANK={local_rank}, but only "\n            f"{visible_gpu_count} CUDA device ordinal(s) are visible."\n        )\n\n    torch.cuda.set_device(local_rank)\n    device = torch.device("cuda", local_rank)\n\n    print(\n        f"[GPU mapping] rank={rank} → cuda:{local_rank} | "\n        f"{torch.cuda.get_device_name(local_rank)}",\n        flush=True,\n    )\n\n    if distributed:\n        dist.init_process_group(\n            backend="nccl",\n            init_method="env://",\n            rank=rank,\n            world_size=world_size,\n        )\n\n    return (\n        distributed,\n        rank,\n        world_size,\n        local_rank,\n        device,\n    )\n\n\ndef cleanup_distributed(distributed):\n    if distributed and dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef reduce_training_stats(\n    loss_sum,\n    correct,\n    n,\n    device,\n    distributed,\n):\n    values = torch.tensor(\n        [loss_sum, correct, n],\n        dtype=torch.float64,\n        device=device,\n    )\n\n    if distributed:\n        dist.all_reduce(\n            values,\n            op=dist.ReduceOp.SUM,\n        )\n\n    loss_total, correct_total, n_total = values.tolist()\n\n    return (\n        loss_total / n_total,\n        correct_total / n_total,\n        int(n_total),\n    )\n\n\ndef binary_metrics(y_true, y_pred, scores):\n    tn, fp, fn, tp = confusion_matrix(\n        y_true,\n        y_pred,\n        labels=[0, 1],\n    ).ravel()\n\n    return {\n        "accuracy": (tp + tn) / (tp + tn + fp + fn),\n        "precision": precision_score(\n            y_true,\n            y_pred,\n            zero_division=0,\n        ),\n        "recall": recall_score(\n            y_true,\n            y_pred,\n            zero_division=0,\n        ),\n        "specificity": (\n            tn / (tn + fp)\n            if (tn + fp)\n            else np.nan\n        ),\n        "f1": f1_score(\n            y_true,\n            y_pred,\n            zero_division=0,\n        ),\n        "auroc": roc_auc_score(y_true, scores),\n        "auprc": average_precision_score(y_true, scores),\n        "true_negative": int(tn),\n        "false_positive": int(fp),\n        "false_negative": int(fn),\n        "true_positive": int(tp),\n    }\n'

DDP_COMMON_SCRIPT.write_text(DDP_COMMON_SOURCE)

py_compile.compile(
    str(DDP_COMMON_SCRIPT),
    doraise=True,
)

print("✅ Python syntax valid:", DDP_COMMON_SCRIPT)

# 8. Write the Large Custom-Transformer Training Program

In [ ]:
# 🔒 RUN ONLY — write custom_all_tokenizers.py

TRAIN_SCRIPT = (
    SCRIPTS_DIR
    / "custom_all_tokenizers.py"
)

TRAIN_SCRIPT_SOURCE = '\nimport argparse\nimport collections\nimport itertools\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import roc_auc_score, average_precision_score\n\nfrom transformers import get_linear_schedule_with_warmup\n\nfrom ddp_common import (\n    setup_distributed,\n    cleanup_distributed,\n    reduce_training_stats,\n    binary_metrics,\n)\n\n\nPAD_ID = 0\nSINGLE_NUC_VOCAB = {\n    "A": 1,\n    "C": 2,\n    "G": 3,\n    "T": 4,\n}\nSINGLE_NUC_VOCAB_SIZE = 5\nONE_HOT_DIM = 4\nK = 6\n\nKMER_VOCAB = {\n    "".join(chars): i + 1\n    for i, chars in enumerate(\n        itertools.product("ACGT", repeat=K)\n    )\n}\nKMER_VOCAB_SIZE = len(KMER_VOCAB) + 1\n\nTOKENIZER_ORDER = [\n    "single_nucleotide",\n    "one_hot",\n    "overlap_6mer",\n    "nonoverlap_6mer",\n    "bpe",\n]\n\n\ndef clean_split(data_dir, seed):\n    data_dir = Path(data_dir)\n\n    seqs = [\n        line.strip().upper()\n        for line in open(data_dir / "seqs.txt")\n        if line.strip()\n    ]\n\n    labels = [\n        int(line.strip())\n        for line in open(data_dir / "labels.txt")\n        if line.strip()\n    ]\n\n    df = pd.DataFrame({\n        "sequence": seqs,\n        "label": labels,\n    })\n\n    df["length"] = df["sequence"].str.len()\n    expected_length = int(df["length"].mode().iloc[0])\n\n    valid = (\n        df["length"].eq(expected_length)\n        & df["sequence"].apply(\n            lambda seq: set(seq) <= set("ACGT")\n        )\n    )\n\n    conflicts = set(\n        df.groupby("sequence")["label"]\n        .nunique()\n        .loc[lambda values: values > 1]\n        .index\n    )\n\n    clean_df = (\n        df[\n            valid\n            & ~df["sequence"].isin(conflicts)\n        ]\n        .drop_duplicates("sequence")\n        .reset_index(drop=True)\n    )\n\n    train_df, val_df = train_test_split(\n        clean_df,\n        test_size=0.20,\n        random_state=seed,\n        stratify=clean_df["label"],\n    )\n\n    return (\n        clean_df,\n        train_df.reset_index(drop=True),\n        val_df.reset_index(drop=True),\n    )\n\n\ndef tokenize_single_nucleotide(seq):\n    return [\n        SINGLE_NUC_VOCAB[base]\n        for base in seq\n    ]\n\n\ndef tokenize_one_hot(seq):\n    mapping = {\n        "A": [1.0, 0.0, 0.0, 0.0],\n        "C": [0.0, 1.0, 0.0, 0.0],\n        "G": [0.0, 0.0, 1.0, 0.0],\n        "T": [0.0, 0.0, 0.0, 1.0],\n    }\n\n    return [\n        mapping[base]\n        for base in seq\n    ]\n\n\ndef tokenize_overlap_6mer(seq):\n    return [\n        KMER_VOCAB[seq[i:i+K]]\n        for i in range(len(seq) - K + 1)\n    ]\n\n\ndef tokenize_nonoverlap_6mer(seq):\n    usable_length = (len(seq) // K) * K\n\n    return [\n        KMER_VOCAB[seq[i:i+K]]\n        for i in range(0, usable_length, K)\n    ]\n\n\ndef train_bpe(sequences, num_merges=80):\n    corpus = [list(seq) for seq in sequences]\n    rules = []\n\n    for _ in range(num_merges):\n        pair_counts = collections.Counter()\n\n        for tokens in corpus:\n            pair_counts.update(\n                zip(tokens[:-1], tokens[1:])\n            )\n\n        if not pair_counts:\n            break\n\n        best_pair = pair_counts.most_common(1)[0][0]\n        merged_token = "".join(best_pair)\n        new_corpus = []\n\n        for tokens in corpus:\n            new_tokens = []\n            i = 0\n\n            while i < len(tokens):\n                if (\n                    i < len(tokens) - 1\n                    and tokens[i] == best_pair[0]\n                    and tokens[i + 1] == best_pair[1]\n                ):\n                    new_tokens.append(merged_token)\n                    i += 2\n                else:\n                    new_tokens.append(tokens[i])\n                    i += 1\n\n            new_corpus.append(new_tokens)\n\n        corpus = new_corpus\n        rules.append(best_pair)\n\n    return rules\n\n\ndef apply_bpe(seq, rules):\n    tokens = list(seq)\n\n    for pair in rules:\n        merged_token = "".join(pair)\n        new_tokens = []\n        i = 0\n\n        while i < len(tokens):\n            if (\n                i < len(tokens) - 1\n                and tokens[i] == pair[0]\n                and tokens[i + 1] == pair[1]\n            ):\n                new_tokens.append(merged_token)\n                i += 2\n            else:\n                new_tokens.append(tokens[i])\n                i += 1\n\n        tokens = new_tokens\n\n    return tokens\n\n\ndef build_tokenizer_info(name, train_sequences, bpe_merges):\n    if name == "single_nucleotide":\n        return {\n            "name": name,\n            "fn": tokenize_single_nucleotide,\n            "continuous": False,\n            "vocab_size": SINGLE_NUC_VOCAB_SIZE,\n            "input_dim": None,\n            "position_capacity": 200,\n        }\n\n    if name == "one_hot":\n        return {\n            "name": name,\n            "fn": tokenize_one_hot,\n            "continuous": True,\n            "vocab_size": None,\n            "input_dim": ONE_HOT_DIM,\n            "position_capacity": 200,\n        }\n\n    if name == "overlap_6mer":\n        return {\n            "name": name,\n            "fn": tokenize_overlap_6mer,\n            "continuous": False,\n            "vocab_size": KMER_VOCAB_SIZE,\n            "input_dim": None,\n            "position_capacity": 195,\n        }\n\n    if name == "nonoverlap_6mer":\n        return {\n            "name": name,\n            "fn": tokenize_nonoverlap_6mer,\n            "continuous": False,\n            "vocab_size": KMER_VOCAB_SIZE,\n            "input_dim": None,\n            "position_capacity": 33,\n        }\n\n    if name == "bpe":\n        rules = train_bpe(\n            train_sequences,\n            num_merges=bpe_merges,\n        )\n\n        tokens_seen = set()\n\n        for seq in train_sequences:\n            tokens_seen.update(\n                apply_bpe(seq, rules)\n            )\n\n        vocab = {\n            token: i + 2\n            for i, token in enumerate(sorted(tokens_seen))\n        }\n\n        def tokenize_bpe(seq):\n            return [\n                vocab.get(token, 1)\n                for token in apply_bpe(seq, rules)\n            ]\n\n        return {\n            "name": name,\n            "fn": tokenize_bpe,\n            "continuous": False,\n            "vocab_size": len(vocab) + 2,\n            "input_dim": None,\n            "position_capacity": 200,\n            "bpe_rules": rules,\n            "bpe_vocab": vocab,\n        }\n\n    raise ValueError(f"Unknown tokenizer: {name}")\n\n\nclass PreTokenizedDataset(Dataset):\n    def __init__(self, seqs, labels, tokenizer_info):\n        self.continuous = tokenizer_info["continuous"]\n        self.inputs = []\n\n        for seq in seqs:\n            tokens = tokenizer_info["fn"](seq)\n\n            if self.continuous:\n                tensor = torch.tensor(\n                    tokens,\n                    dtype=torch.float32,\n                )\n            else:\n                tensor = torch.tensor(\n                    tokens,\n                    dtype=torch.long,\n                )\n\n            self.inputs.append(tensor)\n\n        self.labels = torch.tensor(\n            labels,\n            dtype=torch.long,\n        )\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, idx):\n        return {\n            "input": self.inputs[idx],\n            "label": self.labels[idx],\n        }\n\n\ndef make_collate_fn(tokenizer_info):\n    continuous = tokenizer_info["continuous"]\n    input_dim = tokenizer_info["input_dim"]\n\n    def collate(batch):\n        lengths = [\n            item["input"].shape[0]\n            for item in batch\n        ]\n\n        max_len = max(lengths)\n        batch_size = len(batch)\n\n        if continuous:\n            x_batch = torch.zeros(\n                batch_size,\n                max_len,\n                input_dim,\n                dtype=torch.float32,\n            )\n        else:\n            x_batch = torch.full(\n                (batch_size, max_len),\n                PAD_ID,\n                dtype=torch.long,\n            )\n\n        attention_mask = torch.zeros(\n            batch_size,\n            max_len,\n            dtype=torch.long,\n        )\n\n        labels = torch.zeros(\n            batch_size,\n            dtype=torch.long,\n        )\n\n        for i, item in enumerate(batch):\n            length = item["input"].shape[0]\n            x_batch[i, :length] = item["input"]\n            attention_mask[i, :length] = 1\n            labels[i] = item["label"]\n\n        return {\n            "input": x_batch,\n            "attention_mask": attention_mask,\n            "label": labels,\n        }\n\n    return collate\n\n\nclass AttentionHead(nn.Module):\n    def __init__(self, n_embd, head_size, dropout):\n        super().__init__()\n\n        self.query = nn.Linear(n_embd, head_size, bias=False)\n        self.key = nn.Linear(n_embd, head_size, bias=False)\n        self.value = nn.Linear(n_embd, head_size, bias=False)\n        self.dropout = nn.Dropout(dropout)\n        self.scale = head_size ** -0.5\n\n    def forward(self, x, attention_mask):\n        q = self.query(x)\n        k = self.key(x)\n        v = self.value(x)\n\n        scores = q @ k.transpose(-2, -1)\n        scores = scores * self.scale\n\n        key_mask = attention_mask.unsqueeze(1)\n        scores = scores.masked_fill(\n            key_mask == 0,\n            float("-inf"),\n        )\n\n        weights = F.softmax(scores, dim=-1)\n        weights = self.dropout(weights)\n\n        return weights @ v\n\n\nclass MultiHeadAttention(nn.Module):\n    def __init__(self, n_embd, n_head, dropout):\n        super().__init__()\n\n        assert n_embd % n_head == 0\n        head_size = n_embd // n_head\n\n        self.heads = nn.ModuleList([\n            AttentionHead(\n                n_embd,\n                head_size,\n                dropout,\n            )\n            for _ in range(n_head)\n        ])\n\n        self.projection = nn.Linear(n_embd, n_embd)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x, attention_mask):\n        combined = torch.cat(\n            [\n                head(x, attention_mask)\n                for head in self.heads\n            ],\n            dim=-1,\n        )\n\n        return self.dropout(\n            self.projection(combined)\n        )\n\n\nclass FeedForward(nn.Module):\n    def __init__(self, n_embd, dropout):\n        super().__init__()\n\n        self.net = nn.Sequential(\n            nn.Linear(n_embd, 4 * n_embd),\n            nn.ReLU(),\n            nn.Linear(4 * n_embd, n_embd),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, n_embd, n_head, dropout):\n        super().__init__()\n\n        self.attention = MultiHeadAttention(\n            n_embd,\n            n_head,\n            dropout,\n        )\n        self.feed_forward = FeedForward(\n            n_embd,\n            dropout,\n        )\n        self.norm1 = nn.LayerNorm(n_embd)\n        self.norm2 = nn.LayerNorm(n_embd)\n\n    def forward(self, x, attention_mask):\n        x = x + self.attention(\n            self.norm1(x),\n            attention_mask,\n        )\n\n        x = x + self.feed_forward(\n            self.norm2(x)\n        )\n\n        return x\n\n\nclass DNATransformer(nn.Module):\n    def __init__(\n        self,\n        position_capacity,\n        n_embd,\n        n_head,\n        n_layer,\n        dropout,\n        vocab_size=None,\n        input_dim=None,\n    ):\n        super().__init__()\n\n        assert (vocab_size is None) != (input_dim is None)\n\n        self.continuous = input_dim is not None\n\n        if self.continuous:\n            self.token_embedding = nn.Linear(\n                input_dim,\n                n_embd,\n            )\n        else:\n            self.token_embedding = nn.Embedding(\n                vocab_size,\n                n_embd,\n                padding_idx=PAD_ID,\n            )\n\n        self.position_embedding = nn.Embedding(\n            position_capacity,\n            n_embd,\n        )\n\n        self.blocks = nn.ModuleList([\n            TransformerBlock(\n                n_embd,\n                n_head,\n                dropout,\n            )\n            for _ in range(n_layer)\n        ])\n\n        self.final_norm = nn.LayerNorm(n_embd)\n        self.classifier = nn.Linear(n_embd, 2)\n\n    def forward(self, x, attention_mask):\n        _, token_count = x.shape[:2]\n\n        token_vectors = self.token_embedding(x)\n\n        positions = torch.arange(\n            token_count,\n            device=x.device,\n        )\n\n        h = token_vectors + self.position_embedding(positions)\n\n        for block in self.blocks:\n            h = block(h, attention_mask)\n\n        h = self.final_norm(h)\n\n        mask = attention_mask.unsqueeze(-1).float()\n        pooled = (\n            (h * mask).sum(dim=1)\n            / mask.sum(dim=1).clamp(min=1)\n        )\n\n        return self.classifier(pooled)\n\n\ndef build_optimizer(model, learning_rate, weight_decay):\n    try:\n        optimizer = torch.optim.AdamW(\n            model.parameters(),\n            lr=learning_rate,\n            weight_decay=weight_decay,\n            fused=True,\n        )\n        mode = "AdamW fused=True"\n    except (TypeError, RuntimeError):\n        optimizer = torch.optim.AdamW(\n            model.parameters(),\n            lr=learning_rate,\n            weight_decay=weight_decay,\n        )\n        mode = "AdamW standard"\n\n    return optimizer, mode\n\n\ndef train_epoch(\n    model,\n    optimizer,\n    scheduler,\n    loader,\n    sampler,\n    epoch,\n    device,\n    distributed,\n):\n    model.train()\n\n    if sampler is not None:\n        sampler.set_epoch(epoch)\n\n    loss_sum = 0.0\n    correct = 0\n    n = 0\n\n    for batch_index, batch in enumerate(loader):\n        x = batch["input"].to(\n            device,\n            non_blocking=True,\n        )\n        mask = batch["attention_mask"].to(\n            device,\n            non_blocking=True,\n        )\n        labels = batch["label"].to(\n            device,\n            non_blocking=True,\n        )\n\n        optimizer.zero_grad(set_to_none=True)\n\n        with torch.amp.autocast(\n            device_type="cuda",\n            dtype=torch.bfloat16,\n        ):\n            logits = model(x, mask)\n            loss = F.cross_entropy(logits, labels)\n\n        loss.backward()\n\n        if epoch == 1 and batch_index == 0:\n            missing_grads = [\n                name\n                for name, parameter in model.named_parameters()\n                if parameter.requires_grad and parameter.grad is None\n            ]\n\n            if missing_grads:\n                raise RuntimeError(\n                    "Trainable parameters without gradients: "\n                    + ", ".join(missing_grads)\n                )\n\n        torch.nn.utils.clip_grad_norm_(\n            model.parameters(),\n            max_norm=1.0,\n        )\n\n        optimizer.step()\n        scheduler.step()\n\n        predictions = logits.argmax(dim=1)\n\n        loss_sum += loss.item() * len(labels)\n        correct += (predictions == labels).sum().item()\n        n += len(labels)\n\n    return reduce_training_stats(\n        loss_sum,\n        correct,\n        n,\n        device,\n        distributed,\n    )\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device):\n    model.eval()\n\n    loss_sum = 0.0\n    correct = 0\n    n = 0\n    scores = []\n    predictions = []\n    truth = []\n\n    for batch in loader:\n        x = batch["input"].to(\n            device,\n            non_blocking=True,\n        )\n        mask = batch["attention_mask"].to(\n            device,\n            non_blocking=True,\n        )\n        labels = batch["label"].to(\n            device,\n            non_blocking=True,\n        )\n\n        with torch.amp.autocast(\n            device_type="cuda",\n            dtype=torch.bfloat16,\n        ):\n            logits = model(x, mask)\n            loss = F.cross_entropy(logits, labels)\n\n        pred = logits.argmax(dim=1)\n        prob = torch.softmax(\n            logits.float(),\n            dim=1,\n        )[:, 1]\n\n        loss_sum += loss.item() * len(labels)\n        correct += (pred == labels).sum().item()\n        n += len(labels)\n\n        scores.extend(prob.cpu().numpy())\n        predictions.extend(pred.cpu().numpy())\n        truth.extend(labels.cpu().numpy())\n\n    return {\n        "loss": loss_sum / n,\n        "accuracy": correct / n,\n        "scores": np.asarray(scores),\n        "predictions": np.asarray(predictions),\n        "true": np.asarray(truth),\n    }\n\n\ndef train_one_tokenizer(\n    tokenizer_name,\n    clean_df,\n    train_df,\n    val_df,\n    args,\n    distributed,\n    rank,\n    world_size,\n    device,\n):\n    if distributed:\n        dist.barrier()\n\n    if rank == 0:\n        print("\\n" + "=" * 88, flush=True)\n        print(f"TOKENIZER: {tokenizer_name}", flush=True)\n        print("=" * 88, flush=True)\n\n    preprocessing_start = time.time()\n\n    info = build_tokenizer_info(\n        tokenizer_name,\n        train_df["sequence"].tolist(),\n        args.bpe_merges,\n    )\n\n    train_dataset = PreTokenizedDataset(\n        train_df["sequence"].tolist(),\n        train_df["label"].tolist(),\n        info,\n    )\n\n    val_dataset = PreTokenizedDataset(\n        val_df["sequence"].tolist(),\n        val_df["label"].tolist(),\n        info,\n    )\n\n    token_lengths = np.asarray([\n        len(info["fn"](seq))\n        for seq in clean_df["sequence"]\n    ])\n\n    preprocessing_seconds = time.time() - preprocessing_start\n\n    train_sampler = None\n\n    if distributed:\n        train_sampler = DistributedSampler(\n            train_dataset,\n            num_replicas=world_size,\n            rank=rank,\n            shuffle=True,\n            seed=args.seed,\n        )\n\n    collate_fn = make_collate_fn(info)\n\n    # Inputs are already pre-tokenized in memory.\n    # For this small dataset, extra DataLoader worker processes\n    # add overhead instead of useful preprocessing throughput.\n    worker_count = 0\n\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=args.local_batch_size,\n        shuffle=train_sampler is None,\n        sampler=train_sampler,\n        collate_fn=collate_fn,\n        pin_memory=True,\n        num_workers=worker_count,\n        persistent_workers=worker_count > 0,\n    )\n\n    val_loader = DataLoader(\n        val_dataset,\n        batch_size=args.local_batch_size,\n        shuffle=False,\n        collate_fn=collate_fn,\n        pin_memory=True,\n        num_workers=worker_count,\n        persistent_workers=worker_count > 0,\n    )\n\n    # Same initial seed for the same architecture on every rank.\n    torch.manual_seed(args.seed)\n    torch.cuda.manual_seed_all(args.seed)\n\n    model = DNATransformer(\n        position_capacity=info["position_capacity"],\n        n_embd=args.n_embd,\n        n_head=args.n_head,\n        n_layer=args.n_layer,\n        dropout=args.dropout,\n        vocab_size=info["vocab_size"],\n        input_dim=info["input_dim"],\n    ).to(device)\n\n    total_parameters = sum(\n        p.numel()\n        for p in model.parameters()\n    )\n\n    trainable_parameters = sum(\n        p.numel()\n        for p in model.parameters()\n        if p.requires_grad\n    )\n\n    if trainable_parameters != total_parameters:\n        raise RuntimeError(\n            "Custom Transformer should train all parameters."\n        )\n\n    if distributed:\n        model = DDP(\n            model,\n            device_ids=[device.index],\n            output_device=device.index,\n            gradient_as_bucket_view=True,\n        )\n\n    optimizer, optimizer_mode = build_optimizer(\n        model,\n        args.learning_rate,\n        args.weight_decay,\n    )\n\n    total_steps = len(train_loader) * args.epochs\n    warmup_steps = int(total_steps * args.warmup_ratio)\n\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer,\n        num_warmup_steps=warmup_steps,\n        num_training_steps=total_steps,\n    )\n\n    # Independent dropout streams after DDP model synchronization.\n    torch.manual_seed(args.seed + rank)\n    torch.cuda.manual_seed_all(args.seed + rank)\n\n    if rank == 0:\n        print(f"GPUs: {world_size}", flush=True)\n        print(f"Precision: {args.precision}", flush=True)\n        print(f"Local batch/GPU: {args.local_batch_size}", flush=True)\n        print(\n            f"Global batch: {args.local_batch_size * world_size}",\n            flush=True,\n        )\n        print(\n            f"Mean token count: {token_lengths.mean():.1f}",\n            flush=True,\n        )\n        print(f"Parameters: {total_parameters:,}", flush=True)\n        print(f"Optimizer: {optimizer_mode}", flush=True)\n\n    if distributed:\n        dist.barrier()\n\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats(device)\n    torch.cuda.synchronize(device)\n\n    training_start = time.time()\n    processed_examples = 0\n    history = []\n    final_val = None\n\n    for epoch in range(1, args.epochs + 1):\n        epoch_start = time.time()\n\n        train_loss, train_accuracy, examples_seen = train_epoch(\n            model,\n            optimizer,\n            scheduler,\n            train_loader,\n            train_sampler,\n            epoch,\n            device,\n            distributed,\n        )\n\n        processed_examples += examples_seen\n\n        if distributed:\n            dist.barrier()\n\n        if rank == 0:\n            eval_model = model.module if distributed else model\n\n            final_val = evaluate(\n                eval_model,\n                val_loader,\n                device,\n            )\n\n            val_auroc = roc_auc_score(\n                final_val["true"],\n                final_val["scores"],\n            )\n            val_auprc = average_precision_score(\n                final_val["true"],\n                final_val["scores"],\n            )\n\n            epoch_seconds = time.time() - epoch_start\n\n            history.append({\n                "epoch": epoch,\n                "train_loss": train_loss,\n                "train_accuracy": train_accuracy,\n                "val_loss": final_val["loss"],\n                "val_accuracy": final_val["accuracy"],\n                "val_auroc": val_auroc,\n                "val_auprc": val_auprc,\n                "epoch_seconds": epoch_seconds,\n                "learning_rate": optimizer.param_groups[0]["lr"],\n            })\n\n            print(\n                f"Epoch {epoch}/{args.epochs} | "\n                f"train loss={train_loss:.4f} | "\n                f"train acc={train_accuracy:.3f} | "\n                f"val loss={final_val[\'loss\']:.4f} | "\n                f"AUROC={val_auroc:.4f} | "\n                f"AUPRC={val_auprc:.4f} | "\n                f"{epoch_seconds:.1f}s",\n                flush=True,\n            )\n\n        if distributed:\n            dist.barrier()\n\n    torch.cuda.synchronize(device)\n    training_time = time.time() - training_start\n\n    local_peak_bytes = torch.cuda.max_memory_allocated(device)\n    peak_tensor = torch.tensor(\n        [float(local_peak_bytes)],\n        dtype=torch.float64,\n        device=device,\n    )\n\n    if distributed:\n        dist.all_reduce(\n            peak_tensor,\n            op=dist.ReduceOp.MAX,\n        )\n\n    max_peak_bytes = peak_tensor.item()\n    total_memory_bytes = torch.cuda.get_device_properties(device).total_memory\n\n    result = None\n\n    if rank == 0:\n        history_df = pd.DataFrame(history)\n        best_row = history_df.loc[\n            history_df["val_auroc"].idxmax()\n        ]\n\n        final_metrics = binary_metrics(\n            final_val["true"],\n            final_val["predictions"],\n            final_val["scores"],\n        )\n\n        predictions_df = val_df[["sequence", "label"]].copy()\n        predictions_df["predicted_label"] = final_val["predictions"]\n        predictions_df["binding_probability"] = final_val["scores"]\n        predictions_df["correct"] = (\n            predictions_df["label"]\n            == predictions_df["predicted_label"]\n        )\n\n        examples_per_second = processed_examples / training_time\n\n        peak_memory_gb = max_peak_bytes / (1024 ** 3)\n        total_memory_gb = total_memory_bytes / (1024 ** 3)\n        peak_memory_percent = 100.0 * max_peak_bytes / total_memory_bytes\n\n        run_name = f"custom_large_{tokenizer_name}"\n        output_dir = Path(args.output_dir)\n        output_dir.mkdir(parents=True, exist_ok=True)\n        prefix = output_dir / run_name\n\n        history_df.to_csv(\n            str(prefix) + "_history.csv",\n            index=False,\n        )\n        predictions_df.to_csv(\n            str(prefix) + "_predictions.csv",\n            index=False,\n        )\n\n        eval_model = model.module if distributed else model\n        checkpoint_path = str(prefix) + "_final_checkpoint.pt"\n\n        torch.save(\n            {\n                "tokenizer": tokenizer_name,\n                "state_dict": eval_model.state_dict(),\n                "position_capacity": info["position_capacity"],\n                "vocab_size": info["vocab_size"],\n                "input_dim": info["input_dim"],\n                "n_embd": args.n_embd,\n                "n_head": args.n_head,\n                "n_layer": args.n_layer,\n                "dropout": args.dropout,\n                "bpe_rules": info.get("bpe_rules"),\n                "bpe_vocab": info.get("bpe_vocab"),\n            },\n            checkpoint_path,\n        )\n\n        result = {\n            "run_name": run_name,\n            "family": "custom_transformer",\n            "tokenizer": tokenizer_name,\n            "strategy": "large_scratch_ddp",\n            "num_gpus": int(world_size),\n            "precision": args.precision,\n            "n_embd": int(args.n_embd),\n            "n_head": int(args.n_head),\n            "n_layer": int(args.n_layer),\n            "dropout": float(args.dropout),\n            "epochs": int(args.epochs),\n            "local_batch_size": int(args.local_batch_size),\n            "global_batch_size": int(args.local_batch_size * world_size),\n            "learning_rate": float(args.learning_rate),\n            "weight_decay": float(args.weight_decay),\n            "warmup_ratio": float(args.warmup_ratio),\n            "random_seed": int(args.seed),\n            "bpe_merges": int(args.bpe_merges),\n            "vocab_size": (\n                int(info["vocab_size"])\n                if info["vocab_size"] is not None\n                else None\n            ),\n            "input_dim": (\n                int(info["input_dim"])\n                if info["input_dim"] is not None\n                else None\n            ),\n            "position_capacity": int(info["position_capacity"]),\n            "mean_token_count": float(token_lengths.mean()),\n            "median_token_count": float(np.median(token_lengths)),\n            "min_token_count": int(token_lengths.min()),\n            "max_token_count": int(token_lengths.max()),\n            "trainable_parameters": int(trainable_parameters),\n            "total_parameters": int(total_parameters),\n            "best_val_auroc": float(best_row["val_auroc"]),\n            "best_epoch": int(best_row["epoch"]),\n            "final_val_accuracy": float(final_metrics["accuracy"]),\n            "final_val_precision": float(final_metrics["precision"]),\n            "final_val_recall": float(final_metrics["recall"]),\n            "final_val_specificity": float(final_metrics["specificity"]),\n            "final_val_f1": float(final_metrics["f1"]),\n            "final_val_auroc": float(final_metrics["auroc"]),\n            "final_val_auprc": float(final_metrics["auprc"]),\n            "training_time_seconds": float(training_time),\n            "examples_per_second": float(examples_per_second),\n            "preprocessing_seconds": float(preprocessing_seconds),\n            "peak_gpu_memory_gb": float(peak_memory_gb),\n            "gpu_memory_capacity_gb": float(total_memory_gb),\n            "peak_gpu_memory_percent": float(peak_memory_percent),\n            "optimizer": optimizer_mode,\n            "checkpoint_path": checkpoint_path,\n        }\n\n        with open(\n            str(prefix) + "_summary.json",\n            "w",\n        ) as handle:\n            json.dump(result, handle, indent=2)\n\n        print("\\nFINAL TOKENIZER REPORT", flush=True)\n        print(\n            f"{tokenizer_name} | best AUROC={result[\'best_val_auroc\']:.4f} | "\n            f"time={training_time:.1f}s | "\n            f"examples/s={examples_per_second:.1f} | "\n            f"peak memory={peak_memory_gb:.2f} GB",\n            flush=True,\n        )\n\n    if distributed:\n        dist.barrier()\n\n    # Release this model before creating the next tokenizer model.\n    del model\n    del optimizer\n    del scheduler\n    del train_loader\n    del val_loader\n    del train_dataset\n    del val_dataset\n\n    torch.cuda.empty_cache()\n\n    if distributed:\n        dist.barrier()\n\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument("--data_dir", required=True)\n    parser.add_argument("--output_dir", required=True)\n    parser.add_argument("--n_embd", type=int, required=True)\n    parser.add_argument("--n_head", type=int, required=True)\n    parser.add_argument("--n_layer", type=int, required=True)\n    parser.add_argument("--dropout", type=float, required=True)\n    parser.add_argument("--epochs", type=int, required=True)\n    parser.add_argument("--local_batch_size", type=int, required=True)\n    parser.add_argument("--learning_rate", type=float, required=True)\n    parser.add_argument("--weight_decay", type=float, required=True)\n    parser.add_argument("--warmup_ratio", type=float, required=True)\n    parser.add_argument("--precision", choices=["bf16"], default="bf16")\n    parser.add_argument("--bpe_merges", type=int, default=80)\n    parser.add_argument("--seed", type=int, default=42)\n\n    args = parser.parse_args()\n\n    if args.n_embd % args.n_head != 0:\n        raise ValueError("n_embd must be divisible by n_head")\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU required")\n\n    if not torch.cuda.is_bf16_supported():\n        raise RuntimeError("Notebook 3B expects BF16-capable GPUs")\n\n    (\n        distributed,\n        rank,\n        world_size,\n        local_rank,\n        device,\n    ) = setup_distributed()\n\n    try:\n        np.random.seed(args.seed)\n        torch.manual_seed(args.seed)\n        torch.cuda.manual_seed_all(args.seed)\n        torch.set_float32_matmul_precision("high")\n\n        clean_df, train_df, val_df = clean_split(\n            args.data_dir,\n            args.seed,\n        )\n\n        if rank == 0:\n            print("\\n=== NOTEBOOK 3B LARGE CUSTOM TRANSFORMER ===", flush=True)\n            print(f"Clean sequences: {len(clean_df)}", flush=True)\n            print(f"Train: {len(train_df)} | Validation: {len(val_df)}", flush=True)\n            print(f"GPUs: {world_size}", flush=True)\n            print(\n                f"Architecture: n_embd={args.n_embd}, "\n                f"n_head={args.n_head}, n_layer={args.n_layer}",\n                flush=True,\n            )\n            print(f"Local batch/GPU: {args.local_batch_size}", flush=True)\n            print(f"Global batch: {args.local_batch_size * world_size}", flush=True)\n            print(f"Tokenizers: {TOKENIZER_ORDER}", flush=True)\n\n        combined_results = []\n\n        for tokenizer_name in TOKENIZER_ORDER:\n            result = train_one_tokenizer(\n                tokenizer_name,\n                clean_df,\n                train_df,\n                val_df,\n                args,\n                distributed,\n                rank,\n                world_size,\n                device,\n            )\n\n            if rank == 0:\n                combined_results.append(result)\n\n        if rank == 0:\n            combined_df = pd.DataFrame(combined_results)\n            combined_path = Path(args.output_dir) / "custom_all_tokenizers_summary.csv"\n            combined_df.to_csv(combined_path, index=False)\n\n            print("\\n" + "=" * 88, flush=True)\n            print("ALL TOKENIZERS COMPLETED", flush=True)\n            print("=" * 88, flush=True)\n            print(\n                combined_df[\n                    [\n                        "tokenizer",\n                        "mean_token_count",\n                        "total_parameters",\n                        "best_val_auroc",\n                        "training_time_seconds",\n                        "examples_per_second",\n                        "peak_gpu_memory_gb",\n                    ]\n                ].to_string(index=False),\n                flush=True,\n            )\n            print(f"\\nSaved combined table: {combined_path}", flush=True)\n\n    finally:\n        cleanup_distributed(distributed)\n\n\nif __name__ == "__main__":\n    main()\n'

TRAIN_SCRIPT.write_text(
    TRAIN_SCRIPT_SOURCE
)

py_compile.compile(
    str(TRAIN_SCRIPT),
    doraise=True,
)

print("✅ Python syntax valid:", TRAIN_SCRIPT)


## What the training program does

The single distributed job performs:

```text
create identical cleaned split
↓
train single-nucleotide model
↓
save results + checkpoint
↓
release GPU memory
↓
train one-hot model
↓
save
↓
overlapping 6-mer
↓
non-overlapping 6-mer
↓
BPE
↓
combined comparison table
```

Every tokenizer receives a **fresh randomly initialized Transformer**.

No tokenizer inherits weights from another tokenizer.

For the BPE model, the saved checkpoint also contains the learned **BPE merge rules and BPE vocabulary**, so the representation can be reconstructed later.

# 9. What Is Held Constant?

For all five tokenizer runs:

```text
same cleaned DNA sequences
same 80/20 stratified split
same random seed
same embedding dimension
same attention-head count
same Transformer-layer count
same dropout
same local batch/GPU
same optimizer family
same learning rate
same epoch count
same BF16 precision
same GPU count
```

That is important because it makes tokenizer representation the main experimental variable.

### One unavoidable parameter-count difference

Discrete tokenizers use an embedding table whose size depends on vocabulary size.

Therefore total parameter count can differ slightly between tokenizers even though the Transformer core is identical.

That difference is real and should be reported rather than hidden.

# 10. SLURM Helpers

In [ ]:
# 🔒 RUN ONLY — shell validation + job monitoring

def validate_shell_script(path):
    result = subprocess.run(
        ["bash", "-n", str(path)],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print(result.stderr)
        return False

    print("✅ Shell syntax valid:", path.name)
    return True


def submit_and_stream(
    script_path,
    job_name,
    poll_seconds=2.0,
):
    # A NERSC Jupyter session may itself have a CUDA mask.
    # Do not pass that mask into this new batch allocation.
    submit_env = os.environ.copy()

    inherited_gpu_env = {
        name: submit_env.get(name)
        for name in [
            "CUDA_VISIBLE_DEVICES",
            "NVIDIA_VISIBLE_DEVICES",
            "ROCR_VISIBLE_DEVICES",
            "GPU_DEVICE_ORDINAL",
        ]
        if name in submit_env
    }

    for name in [
        "CUDA_VISIBLE_DEVICES",
        "NVIDIA_VISIBLE_DEVICES",
        "ROCR_VISIBLE_DEVICES",
        "GPU_DEVICE_ORDINAL",
    ]:
        submit_env.pop(name, None)

    print(
        "Notebook GPU environment:",
        inherited_gpu_env if inherited_gpu_env else "<none>",
    )
    print("Submitting with inherited GPU visibility removed.")

    submit = subprocess.run(
        ["sbatch", str(script_path)],
        capture_output=True,
        text=True,
        env=submit_env,
    )

    if submit.returncode != 0:
        print(submit.stderr)
        raise RuntimeError("sbatch rejected the job")

    job_id = submit.stdout.strip().split()[-1]
    output_file = SLURM_LOG_DIR / f"{job_name}-{job_id}.out"

    print("Submitted job", job_id)
    print("Output:", output_file)

    last_size = 0

    while True:
        if output_file.exists():
            with output_file.open("r") as handle:
                handle.seek(last_size)
                text = handle.read()
                if text:
                    print(text, end="")
                last_size = handle.tell()

        active = subprocess.run(
            ["squeue", "-h", "-j", job_id],
            capture_output=True,
            text=True,
        ).stdout.strip()

        if not active:
            if output_file.exists():
                with output_file.open("r") as handle:
                    handle.seek(last_size)
                    text = handle.read()
                    if text:
                        print(text, end="")
            break

        time.sleep(poll_seconds)

    summary = subprocess.run(
        [
            "sacct",
            "-j",
            job_id,
            "--format=JobID,State,ExitCode,Elapsed,AllocTRES",
            "-n",
            "-P",
        ],
        capture_output=True,
        text=True,
    ).stdout.strip()

    print("\n--- sacct summary ---")
    print(summary)

    main_state = None

    for line in summary.splitlines():
        fields = line.split("|")
        if len(fields) >= 2 and fields[0] == job_id:
            main_state = fields[1]
            break

    if main_state is None or not main_state.startswith("COMPLETED"):
        raise RuntimeError(
            f"SLURM job {job_id} finished with state {main_state}."
        )

    print(f"✅ Job {job_id} completed successfully.")
    return job_id

# 11. Build the Multi-GPU Job

In [ ]:
# 🔒 RUN ONLY — command-line arguments

def build_training_arguments():
    values = [
        "--data_dir", str(DATA_DIR),
        "--output_dir", str(RESULTS_DIR),
        "--n_embd", str(MODEL_CONFIG["n_embd"]),
        "--n_head", str(MODEL_CONFIG["n_head"]),
        "--n_layer", str(MODEL_CONFIG["n_layer"]),
        "--dropout", str(MODEL_CONFIG["dropout"]),
        "--epochs", str(MODEL_CONFIG["epochs"]),
        "--local_batch_size", str(MODEL_CONFIG["local_batch_size"]),
        "--learning_rate", str(MODEL_CONFIG["learning_rate"]),
        "--weight_decay", str(MODEL_CONFIG["weight_decay"]),
        "--warmup_ratio", str(MODEL_CONFIG["warmup_ratio"]),
        "--precision", str(MODEL_CONFIG["precision"]),
        "--bpe_merges", str(MODEL_CONFIG["bpe_merges"]),
        "--seed", str(MODEL_CONFIG["seed"]),
    ]

    return " ".join(
        shlex.quote(value)
        for value in values
    )

In [ ]:
# 🔒 RUN ONLY — generate the Slurm job

def write_training_slurm(
    walltime="01:30:00",
):
    if SLURM_QOS == "shared" and GPU_COUNT > 2:
        raise ValueError(
            "Current shared testing should use 1 or 2 GPUs."
        )

    arguments = build_training_arguments()
    job_name = "nb3b-custom-all-tokenizers"
    path = SCRIPTS_DIR / "custom_all_tokenizers.slurm"

    path.write_text(
f"""#!/bin/bash
#SBATCH -A {NERSC_ACCOUNT}
#SBATCH -C gpu
#SBATCH -q {SLURM_QOS}
#SBATCH -t {walltime}

#SBATCH -N 1
#SBATCH --ntasks-per-node={GPU_COUNT}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-node={GPU_COUNT}
#SBATCH --gpu-bind=none

#SBATCH -J {job_name}
#SBATCH -o {SLURM_LOG_DIR}/{job_name}-%j.out

export SLURM_CPU_BIND="cores"
export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT=$((10000 + SLURM_JOB_ID % 50000))
export NCCL_DEBUG=WARN

echo "[BATCH] CUDA_VISIBLE_DEVICES before cleanup=${{CUDA_VISIBLE_DEVICES-<unset>}}"
echo "[BATCH] SLURM_JOB_GPUS=${{SLURM_JOB_GPUS-<unset>}}"

unset CUDA_VISIBLE_DEVICES
unset NVIDIA_VISIBLE_DEVICES
unset ROCR_VISIBLE_DEVICES
unset GPU_DEVICE_ORDINAL

srun --gpu-bind=none bash -c '
    export RANK=$SLURM_PROCID
    export LOCAL_RANK=$SLURM_LOCALID
    export WORLD_SIZE=$SLURM_NTASKS

    echo "[SLURM→DDP] RANK=$RANK LOCAL_RANK=$LOCAL_RANK WORLD_SIZE=$WORLD_SIZE CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES-<unset>}}"

    {NOTEBOOK_PYTHON} {TRAIN_SCRIPT} {arguments}
'
"""
    )

    if not validate_shell_script(path):
        raise RuntimeError(
            "Fix shell syntax before submitting."
        )

    return path, job_name

In [ ]:
# ✏️ RUN THIS — generate and inspect the job

TRAIN_SLURM_SCRIPT, TRAIN_JOB_NAME = (
    write_training_slurm(
        walltime="01:30:00"
    )
)

print(TRAIN_SLURM_SCRIPT.read_text())

### ✅ CHECKPOINT — before submission

For a 2-GPU run, the printed script should contain:

```bash
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-node=2
#SBATCH --gpu-bind=none
```

and:

```bash
RANK=$SLURM_PROCID
LOCAL_RANK=$SLURM_LOCALID
WORLD_SIZE=$SLURM_NTASKS
```

At runtime we want:

```text
rank 0 → cuda:0
rank 1 → cuda:1
```

# 12. Train All Five Tokenizers

In [ ]:
# ✏️ RUN THIS

TRAIN_JOB_ID = submit_and_stream(
    TRAIN_SLURM_SCRIPT,
    TRAIN_JOB_NAME,
)

### What the log should show

For each tokenizer:

```text
TOKENIZER: single_nucleotide
...
FINAL TOKENIZER REPORT

TOKENIZER: one_hot
...
FINAL TOKENIZER REPORT

TOKENIZER: overlap_6mer
...

TOKENIZER: nonoverlap_6mer
...

TOKENIZER: bpe
...
```

At the very end:

```text
ALL TOKENIZERS COMPLETED
```

The script saves each model before moving to the next one.

# 13. Load the Combined Results

In [ ]:
# 🔒 RUN ONLY

COMBINED_RESULTS_PATH = (
    RESULTS_DIR
    / "custom_all_tokenizers_summary.csv"
)

if not COMBINED_RESULTS_PATH.exists():
    raise FileNotFoundError(
        "The all-tokenizer training job has not produced "
        "the combined result table yet."
    )

results = pd.read_csv(
    COMBINED_RESULTS_PATH
)

results

In [ ]:
# 👀 READ — compact comparison table

comparison_columns = [
    "tokenizer",
    "mean_token_count",
    "total_parameters",
    "best_val_auroc",
    "final_val_auprc",
    "training_time_seconds",
    "examples_per_second",
    "peak_gpu_memory_gb",
]

results[
    comparison_columns
].sort_values(
    "best_val_auroc",
    ascending=False,
)

# 14. Biological Performance Across Tokenizers

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values(
    "best_val_auroc",
    ascending=False,
)

plt.bar(
    ordered["tokenizer"],
    ordered["best_val_auroc"],
)

plt.ylabel("Best validation AUROC")
plt.xlabel("Tokenizer")
plt.title("Large Custom Transformer — AUROC by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values(
    "final_val_auprc",
    ascending=False,
)

plt.bar(
    ordered["tokenizer"],
    ordered["final_val_auprc"],
)

plt.ylabel("Final validation AUPRC")
plt.xlabel("Tokenizer")
plt.title("Large Custom Transformer — AUPRC by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# 15. Computational Cost Across Tokenizers

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values(
    "training_time_seconds",
)

plt.bar(
    ordered["tokenizer"],
    ordered["training_time_seconds"],
)

plt.ylabel("Training time (seconds)")
plt.xlabel("Tokenizer")
plt.title("Training Time by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values(
    "examples_per_second",
    ascending=False,
)

plt.bar(
    ordered["tokenizer"],
    ordered["examples_per_second"],
)

plt.ylabel("Training examples / second")
plt.xlabel("Tokenizer")
plt.title("GPU Training Throughput by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    results["mean_token_count"],
    results["training_time_seconds"],
    s=80,
)

for _, row in results.iterrows():
    plt.annotate(
        row["tokenizer"],
        (
            row["mean_token_count"],
            row["training_time_seconds"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel("Mean token count")
plt.ylabel("Training time (seconds)")
plt.title("Token Count vs Training Time")
plt.tight_layout()
plt.show()

### What should you look for?

If tokenizer length affects compute as expected, representations with fewer tokens should often require less attention work.

But remember:

```text
shorter sequence ≠ automatically better biology
```

A tokenizer can be computationally cheap while losing useful sequence information.

The best representation should balance:

```text
predictive performance
+
computational efficiency
```

# 16. GPU Memory Across Tokenizers

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values(
    "peak_gpu_memory_gb",
)

plt.bar(
    ordered["tokenizer"],
    ordered["peak_gpu_memory_gb"],
)

plt.ylabel("Peak allocated GPU memory (GB)")
plt.xlabel("Tokenizer")
plt.title("Peak GPU Memory by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Why can memory differ if the architecture is the same?

The Transformer core is fixed, but tokenizers change:

- sequence length,
- attention tensor size,
- discrete vocabulary embedding size,
- dynamic-padding length.

So representation affects both **activations** and, for discrete tokenizers, part of the **parameter count**.

# 17. Model Parameter Counts

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values(
    "total_parameters",
)

plt.bar(
    ordered["tokenizer"],
    ordered["total_parameters"],
)

plt.ylabel("Total trainable parameters")
plt.xlabel("Tokenizer")
plt.title("Parameter Count by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Why are the parameter counts not identical?

The **Transformer blocks are identical**.

However:

```text
single nucleotide → small embedding table
6-mer            → larger embedding table
BPE              → learned vocabulary-sized embedding table
one-hot          → linear projection instead of token embedding
```

That input layer changes the total parameter count slightly.

Students should report this rather than claiming that the models have perfectly identical parameter counts.

# 18. Performance vs Compute Tradeoff

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    results["training_time_seconds"],
    results["best_val_auroc"],
    s=80,
)

for _, row in results.iterrows():
    plt.annotate(
        row["tokenizer"],
        (
            row["training_time_seconds"],
            row["best_val_auroc"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel("Training time (seconds)")
plt.ylabel("Best validation AUROC")
plt.title("Performance vs Computational Cost")
plt.tight_layout()
plt.show()

## Interpreting this graph

An attractive tokenizer would move toward:

```text
higher AUROC
+
lower training time
```

There may not be one tokenizer that dominates both dimensions.

That is a real research result.

# 19. Inspect One Tokenizer in Detail

In [ ]:
# ✏️ EDIT ME

TOKENIZER_TO_INSPECT = "overlap_6mer"

HISTORY_PATH = (
    RESULTS_DIR
    / f"custom_large_{TOKENIZER_TO_INSPECT}_history.csv"
)

history = pd.read_csv(HISTORY_PATH)

history

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history["epoch"],
    history["train_loss"],
    marker="o",
    label="Train loss",
)

plt.plot(
    history["epoch"],
    history["val_loss"],
    marker="o",
    label="Validation loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{TOKENIZER_TO_INSPECT} — Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history["epoch"],
    history["val_auroc"],
    marker="o",
)

plt.xlabel("Epoch")
plt.ylabel("Validation AUROC")
plt.title(f"{TOKENIZER_TO_INSPECT} — AUROC")
plt.tight_layout()
plt.show()

# 20. Student Analysis Questions

1. Which tokenizer produced the highest validation AUROC?
2. Which tokenizer produced the highest AUPRC?
3. Which tokenizer trained fastest?
4. Which tokenizer processed the most examples per second?
5. Which tokenizer used the fewest tokens per DNA sequence?
6. Did the tokenizer with the fewest tokens also train fastest?
7. Which tokenizer used the most GPU memory?
8. Why do overlapping 6-mers and non-overlapping 6-mers behave so differently computationally?
9. Why can one-hot and single-nucleotide inputs have similar token counts but different input layers?
10. Did BPE provide a useful compromise between sequence length and model performance?
11. Which tokenizer would you choose if AUROC were the only goal?
12. Which tokenizer would you choose if compute time were limited?
13. Which tokenizer gives the best overall compromise for this dataset?

# 21. Notebook 3B Summary

Notebook 3B scales the custom Transformer from Notebook 2 into a multi-GPU experiment.

The workflow is:

```text
same cleaned CTCF dataset
↓
same train/validation split
↓
five DNA tokenizers
↓
five fresh large custom Transformers
↓
BF16 + DDP on every tokenizer run
↓
save five checkpoints
↓
compare biology + compute
```

The central scientific question is:

> **How does DNA representation change both predictive performance and computational cost when the Transformer architecture is held constant?**

The central HPC question is:

> **How do token count, dynamic padding, and vocabulary representation affect GPU training time, throughput, and memory?**

# ✅ Notebook 3A + 3B Together

The two capstones now complement one another:

```text
Notebook 3A
Pretrained DNABERT
→ full useful fine-tuning
→ maximize trainable pretrained parameters
→ multi-GPU DDP

Notebook 3B
Custom Transformer from scratch
→ larger architecture
→ all five tokenizer methods
→ every model parameter trainable
→ multi-GPU DDP
```

Together they let students compare:

```text
pretrained genomic language model
vs
from-scratch Transformer
```

while still learning how to use HPC resources for long GPU training workloads.

# ✅ End of Notebook 3B